In [ ]:
import pandas as pd
pd.options.mode.chained_assignment = None 
import warnings#
import pickle
warnings.filterwarnings("ignore")

----------------------------------------------
----------------------------------------------
----------------------------------------------

## Test Cohorts

old cohort did not exclude nf and aplasia

In [ ]:
#NF
old_cohort = pd.read_csv("/Users/zy51nise/Documents/BIONETs/FLabNet/Code_main/FLabBench-pipeline/saved_data/cohorts/LIT/cohort_neutropenic_fever.csv.gz")
new_cohort = pd.read_csv("/Users/zy51nise/Documents/BIONETs/FLabNet/Code_main/flabnet-pipeline/MIMIC_IV/saved_data/cohorts/mimic_cohort_NF_30_days.csv.gz")

In [ ]:
# MIMIC_all
#(mimic all matches when we have not excluded the nf and aplasia cohorts)
old_cohort =pd.read_csv("/Users/zy51nise/Documents/BIONETs/FLabNet/Code_main/flabnet-pipeline/MIMIC_IV/saved_data/cohorts/mimic_all.csv.gz",compression='gzip')
new_cohort = pd.read_csv("/Users/zy51nise/Documents/BIONETs/FLabNet/Code_main/FLabBench-pipeline/saved_data/cohorts/LIT/cohort_mimic_all.csv.gz",compression="gzip")

In [ ]:
nf_cohort = pd.read_csv("/Users/zy51nise/Documents/BIONETs/FLabNet/Code_main/FLabBench-pipeline/saved_data/cohorts/LIT/cohort_neutropenic_fever.csv.gz")
aplasia_cohort =pd.read_csv("/Users/zy51nise/Documents/BIONETs/FLabNet/Code_main/FLabBench-pipeline/saved_data/cohorts/LIT/cohort_aplasia.csv.gz")

subs = np.concatenate([nf_cohort.subject_id.unique() , aplasia_cohort.subject_id.unique()])
cohort_subs = set(subs)
old_cohort = old_cohort[~old_cohort["subject_id"].isin(cohort_subs)]

In [ ]:
print(old_cohort.subject_id.nunique(),new_cohort.subject_id.nunique())
#print(old_cohort.label.value_counts(),new_cohort.label.value_counts())
print(old_cohort.hadm_id.nunique(),new_cohort.hadm_id.nunique())
print(set(old_cohort.hadm_id) - set(new_cohort.hadm_id))

----------------------------------------------
----------------------------------------------
----------------------------------------------

## Test features

In [ ]:
#mimic_all
old_features = pd.read_csv(f"/Users/zy51nise/Documents/BIONETs/FLabNet/Code_main/flabnet-pipeline/MIMIC_IV/saved_data/processed_admission_features_for_ts/mimic_all/mimic_all_admissions_labs_14_days_to_ts.csv.gz")
new_features = pd.read_csv(f"/Users/zy51nise/Documents/BIONETs/FLabNet/Code_main/FLabBench-pipeline/saved_data/features/mimic_all/features.csv.gz")
old_features = old_features[~old_features["subject_id"].isin(cohort_subs)]

#mimic_nf
#old_features = pd.read_csv(f"/Users/zy51nise/Documents/BIONETs/FLabNet/Code_main/flabnet-pipeline/MIMIC_IV/saved_data/features/mimic_cohort_NF_30_days_admissions_labs_14_days.csv.gz")
#new_features = pd.read_csv(f"/Users/zy51nise/Documents/BIONETs/FLabNet/Code_main/FLabBench-pipeline/saved_data/features/neutropenic_fever/features.csv.gz")


#mimic_aplasia
#old_features = pd.read_csv(f"/Users/zy51nise/Documents/BIONETs/FLabNet/Code_main/flabnet-pipeline/MIMIC_IV/saved_data/features/mimic_cohort_aplasia_45_days_admissions_labs_14_days.csv.gz")
#new_features = pd.read_csv(f"/Users/zy51nise/Documents/BIONETs/FLabNet/Code_main/FLabBench-pipeline/saved_data/features/aplasia/features.csv.gz")

print(old_features.subject_id.nunique(),new_features.subject_id.nunique())
print(old_features.itemid.nunique(),new_features.itemid.nunique())
print(old_features.hadm_id.nunique(),new_features.hadm_id.nunique())
print(len(set(old_features.hadm_id) - set(new_features.hadm_id)))

print(len(old_features), len(new_features))

mismatch because the old version contains all itemids not only the top 100


both still contain the NF and aplasia subjects

In [ ]:
with open ("/Users/zy51nise/Documents/BIONETs/FLabNet/Code_main/FLabBench-pipeline/data/top_features/mimic_top100_features.pkl", "rb") as f:
    top_features = pickle.load(f)
top_features = [int(x) for x in top_features]

In [ ]:
old_features_sub = old_features[old_features["itemid"].isin(top_features)]
print(old_features_sub.subject_id.nunique(),new_features.subject_id.nunique())
print(old_features_sub.itemid.nunique(),new_features.itemid.nunique())
print(old_features_sub.hadm_id.nunique(),new_features.hadm_id.nunique())
print(len(set(old_features_sub.hadm_id) - set(new_features.hadm_id)))

print(len(old_features_sub), len(new_features))

----------------------------------------------
----------------------------------------------
----------------------------------------------

# Check Folds

Important:

In old pipeline for getting mimic all folds we had:

1. remove the nf and aplasia subjects
2. remove admissions without feature (but feature selection was not applied)
3. Then split

If I do the same in new pieline the folds match: extract mimic all cohort excluding NF and Aplasia then extract features without feature selection and then split folds in this case they match.

But in new pipeline it makes sense to remove these admissions where they do not have features with feature selection true to avoid ahving admission without measurement after selecting the features. I wdid this for the other cohorts as well.


In [ ]:
fold = 0

with open (f"/Users/zy51nise/Documents/BIONETs/FLabNet/Code_main/flabnet-pipeline/MIMIC_IV/saved_data/folds/mimic_cohort_NF_30_days/fold_{fold}.pkl","rb") as f:
    folds_old =pickle.load(f)

#with open (f"/Users/zy51nise/Documents/BIONETs/FLabNet/Code_main/flabnet-pipeline/MIMIC_IV/saved_data/multi_seed_training/folds/mimic_cohort_aplasia_45_days/seed_42/fold_{fold}.pkl","rb") as f:
#    seed_folds_old = pickle.load(f)

with open(f"/Users/zy51nise/Documents/BIONETs/FLabNet/Code_main/FLabBench-pipeline/saved_data/folds/neutropenic_fever/seed_42/fold_{fold}.pkl", "rb") as f:
    folds_new = pickle.load(f)


In [ ]:
sub_ids_old = set(folds_old[1][:,0])
sub_ids_new = set (folds_new[1][:,0])

print(len(sub_ids_new), len(sub_ids_old))
print(len(sub_ids_new - sub_ids_old))

----------------------------------------------
----------------------------------------------
----------------------------------------------

# Test input dicts

In [ ]:
with open ("/Users/zy51nise/Documents/BIONETs/FLabNet/Code_main/FLabBench-pipeline/saved_data/dicts/neutropenic_fever/random_forest/040526/fold_0/agg_int_24/impute_fill/variant_VMD/input_dict.pkl", "rb") as f:
    input_new=pickle.load(f)

with open ("/Users/zy51nise/Documents/BIONETs/FLabNet/Code_main/flabnet-pipeline/MIMIC_IV/saved_data/training/mimic_cohort_NF_30_days/concatenate/minority/feature_selection_True/feat_type_VMD/agg_interval_24h/Sub_val__fold_1.pkl", "rb") as f:
    old_subs = pickle.load(f)
    
with open ("/Users/zy51nise/Documents/BIONETs/FLabNet/Code_main/flabnet-pipeline/MIMIC_IV/saved_data/training/mimic_cohort_NF_30_days/concatenate/minority/feature_selection_True/feat_type_VMD/agg_interval_24h/X_val__fold_1.pkl", "rb") as f:
    input_old = pickle.load(f)

new_features = pd.read_csv(f"/Users/zy51nise/Documents/BIONETs/FLabNet/Code_main/FLabBench-pipeline/saved_data/features/neutropenic_fever/features.csv.gz")
new_features['int'] = (new_features['minute'] // (60 *24)).astype(int)

In [ ]:
selected_sub = 10101615
adms = new_features[new_features.subject_id ==selected_sub].hadm_id.unique()
print("admissions:", adms)

In [ ]:
selected_adm = 26750486
selected_itemid = '51133'

In [ ]:
#Find indices 
ts_id_to_ind = input_new["ts_id_to_ind"]
var_to_ind   = input_new["var_to_ind"]
feature_names = input_new["feature_names"]
X_flat       = input_new["X_flat"]
#values_norm  = input_new["values_norm"]

ts_ind = ts_id_to_ind[selected_adm]
print("admission index:",ts_ind)

var = selected_itemid
var_ind = var_to_ind [var]
print("itemid index",var_ind)

### Check the new input features and the original features to see if they match

In [ ]:
ind_to_hadm = {v: k for k, v in input_new["ts_id_to_ind"].items()}
hadm_id = ind_to_hadm[ts_ind]
print("hadm_id:",hadm_id)
ind_to_itemid = {v: k for k, v in input_new["var_to_ind"].items()}
itemid = ind_to_itemid[var_ind]
print("itemid:",itemid)



values = input_new["values_raw"][ts_ind,:,var_ind]
obs    = input_new["obs"][ts_ind, :, var_ind]           
delta  = input_new["delta"][ts_ind, :, var_ind]   
X_flat = input_new["X_flat"][ts_ind] 
feature_names = input_new["feature_names"]
demo = input_new["demo_norm"][ts_ind]
print("Age & Gender",demo)

features= new_features[(new_features.hadm_id == hadm_id) & (new_features.itemid == int(selected_itemid))]
#features= new_features[(new_features.hadm_id == hadm_id)&(new_features.itemid == itemid)]
print(features)

print("V",values)
print("M",obs)
print("D",delta)
print("X_flat", len(X_flat))
print("feature_names", len(feature_names))

### Check if feature names are correct

In [ ]:
feature_names = input_new["feature_names"]   # full list, no [ts]
X_flat_ts = input_new["X_flat"][ts_ind]          # patient row
indices = [i for i, name in enumerate(feature_names) if f"{itemid}_V_Bin" in name]
print(indices)
#indices.sort(key=lambda i: int(feature_names[i].split("Bin")[1]))  # ensure Bin0→Bin14 order

print(f"Found {len(indices)} bins for {itemid}_V")
print([feature_names[i] for i in indices])
print(X_flat_ts[indices])

indices = [i for i, name in enumerate(feature_names) if name in ("age", "gender")]
print([feature_names[i] for i in indices])
print(X_flat_ts[indices])

## Check entire inputdict

In [ ]:
old_subs.tolist()
adms = new_features[new_features.subject_id.isin(old_subs)].hadm_id.unique()
ts_id_to_ind = input_new["ts_id_to_ind"]
ts_indices = [ts_id_to_ind[adm] for adm in adms]
print("admission index:",ts_indices)

In [ ]:
#select hadmids in old input
values = input_new["values_raw"][ts_indices,:,:].reshape(len(ts_indices),-1)
obs    = input_new["obs"][ts_indices,:,:].reshape(len(ts_indices),-1)  
delta  = input_new["delta"][ts_indices,:,:].reshape(len(ts_indices),-1)
feature_names = input_new["feature_names"][:4500]
df_values = pd.DataFrame(values, index=[adms])
df_obs =  pd.DataFrame(obs, index=[adms])
df_delta =  pd.DataFrame(delta, index=[adms])

new_df = pd.DataFrame(np.concatenate([df_values, df_obs, df_delta], axis=1), index=[adms], columns=feature_names)

In [ ]:
new_df

In [ ]:
old_df = input_old.iloc[:, :4500]
old_df

In [ ]:
np.allclose(old_df.sum().values, new_df.sum().values, atol=1e-6)

The sum matches. The new input dict inclues all admission not splitted yet, but If I choose the same subjects from old_input and extract those from new input dict they match. 

----------------------------------------------
----------------------------------------------
----------------------------------------------

# Test results

In [ ]:
results = pd.read_csv("/Users/zy51nise/Documents/BIONETs/FLabNet/Code_main/FLabBench-pipeline/saved_data/results/neutropenic_fever/random_forest/050526/fold_4/agg_int_24/impute_fill/variant_VMD/results_final.csv", on_bad_lines='skip')
results

In [ ]:
grid_results = pd.read_csv("/Users/zy51nise/Documents/BIONETs/FLabNet/Code_main/FLabBench-pipeline/saved_data/results/neutropenic_fever/random_forest/050526/fold_1/agg_int_24/impute_fill/variant_VMD/grid_results.csv")
grid_results


----------------------------------------------
----------------------------------------------
----------------------------------------------

In [ ]:
nf_cohort = pd.read_csv("/Users/zy51nise/Documents/BIONETs/FLabNet/Code_main/FLabBench-pipeline/saved_data/cohorts/LIT/cohort_neutropenic_fever.csv.gz")
aplasia_cohort =pd.read_csv("/Users/zy51nise/Documents/BIONETs/FLabNet/Code_main/FLabBench-pipeline/saved_data/cohorts/LIT/cohort_aplasia.csv.gz")

In [ ]:
subs = np.concatenate([nf_cohort.subject_id.unique() , aplasia_cohort.subject_id.unique()])

In [ ]:
cohort_subs = set(subs)

In [ ]:
mimic_sub = old_features[~old_features["subject_id"].isin(cohort_subs)]

In [ ]:
print(old_features_sub.subject_id.nunique(),new_features.subject_id.nunique())
#print(old_cohort.label.value_counts(),new_cohort.label.value_counts())
print(old_features_sub.hadm_id.nunique(),new_features.hadm_id.nunique())
print(len(set(old_features_sub.hadm_id) - set(new_features.hadm_id)))

## Test DTB cohorts

In [ ]:
import pandas as pd
sel_edges = pd.read_csv(r"data/MIMIC_IV/cohorts/DTB/selected_edges_DTB_all.csv")
print(len(sel_edges))

In [ ]:
possible_cohorts = sel_edges[(sel_edges["n_pos"] > 10) & (sel_edges["n_neg"] > 50)]
possible_cohorts["n_cohort"] = possible_cohorts["n_pos"] + possible_cohorts["n_neg"]
possible_cohorts["target_rate"] = possible_cohorts["n_pos"] / possible_cohorts["n_cohort"]
len(possible_cohorts)

select cohort with target-rate > 0.02

In [ ]:
possible_cohorts = possible_cohorts[possible_cohorts["target_rate"] > 0.01]
print(len(possible_cohorts))

In [ ]:
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 6))

plot_df = possible_cohorts.copy()

# Joint plot with marginal histograms
g = sns.jointplot(
    data = plot_df,
    x="n_cohort",
    y="target_rate",
    #hue="RR",
    palette="viridis",
    alpha=0.7,
    kind="scatter",   # scatter in the middle
    marginal_kws=dict(bins=25, fill=True)
)

g.set_axis_labels("Cohort size", "Target frequency(D2 within 5y)")
plt.suptitle("Cohort size vs target frequency with RR-colour", y=1.02)

plt.show()


In [ ]:
plot_df["log_n_cohort"] = np.log10(plot_df["n_cohort"])
plot_df["log_target_rate"] = np.log10(plot_df["target_rate"])

# Joint plot with marginal histograms
g = sns.jointplot(
    data=plot_df,
    x="log_n_cohort",
    y="log_target_rate",
    #hue="RR",
    palette="viridis",
    alpha=0.7,
    kind="scatter",   # scatter in the middle
    marginal_kws=dict(bins=25, fill=True)
)

g.set_axis_labels("log10(Cohort size)", "log10(Target frequency as D2 within 5y)")

plot_df["log_n_cohort"] = np.log10(plot_df["n_cohort"])
plot_df["log_target_rate"] = np.log10(plot_df["target_rate"])



plt.show()

In [ ]:
plot_df["D1cat"] = plot_df["D1"].str[:1]
plot_df["D2cat"] = plot_df["D2"].str[:1]

cmap = sns.color_palette("husl", 16)

plt.figure(figsize=(10, 10))
g = sns.scatterplot(
    data=plot_df,
    x="log_n_cohort",
    y="log_target_rate",
    hue="D1cat",
    palette=cmap,
    alpha=0.7
)

In [ ]:
possible_cohorts["log_n_cohort"] = np.log10(possible_cohorts["n_cohort"])
possible_cohorts["log_target_rate"] = np.log10(possible_cohorts["target_rate"])

In [ ]:
max = possible_cohorts["log_n_cohort"].max()
min = possible_cohorts["log_n_cohort"].min()
step = (max - min) / 4
bins = [min, min+step, min+2*step, min+3*step, min+4*step]
labels = ["very_low", "low", "medium", "high"]
possible_cohorts["target_group"] = pd.cut(possible_cohorts["log_n_cohort"], bins=bins, labels=labels, include_lowest=True)
possible_cohorts.groupby("target_group")[["log_n_cohort"]].describe()


In [ ]:
max = possible_cohorts["log_target_rate"].max()
min = possible_cohorts["log_target_rate"].min()
step = (max - min) / 4
bins = [min, min+step, min+2*step, min+3*step, min+4*step]
labels = ["very_low", "low", "medium", "high"]
possible_cohorts["target_group"] = pd.cut(possible_cohorts["log_target_rate"], bins=bins, labels=labels, include_lowest=True)
possible_cohorts.groupby("target_group")[["log_target_rate"]].describe()

similarity in codes 

In [ ]:
df = possible_cohorts.copy()

In [ ]:
df["D1cat"] = df["D1"].str[:1]
df["D2cat"] = df["D2"].str[:1]

In [ ]:
df

In [ ]:
df.groupby("D1cat")["D2cat"].unique()

df.groupby("D2cat")["D1cat"].unique()


In [ ]:
# 405 unique D1 and 464 unique D2
# 16 unique categories for D1 and 16 unique categories for D2

Certain infectious and parasitic diseases
(A00-B99)
Neoplasms
(C00-D48)
Diseases of the blood and blood-forming organs and certain disorders involving the immune mechanism
(D50-D89)
Endocrine, nutritional and metabolic diseases
(E00-E90)
Mental and behavioural disorders
(F00-F99)
Diseases of the nervous system
(G00-G99)
Diseases of the eye and adnexa
(H00-H59)
Diseases of the ear and mastoid process
(H60-H95)
Diseases of the circulatory system
(I00-I99)
Diseases of the respiratory system
(J00-J99)
Diseases of the digestive system
(K00-K93)
Diseases of the skin and subcutaneous tissue
(L00-L99)
Diseases of the musculoskeletal system and connective tissue
(M00-M99)
Diseases of the genitourinary system
(N00-N99)
Pregnancy, childbirth and the puerperium
(O00-O99)
Certain conditions originating in the perinatal period
(P00-P96)
Congenital malformations, deformations and chromosomal abnormalities
(Q00-Q99)

In [ ]:
MIMIC_IV_PATH = "/Users/zy51nise/Documents/BIONETs/FLabNet/Data/mimiciv/2.0/icu/"
MIMIC_III_PATH = "/Users/zy51nise/Documents/BIONETs/FLabNet/Data/mimiciii/"

In [ ]:
items_iii = pd.read_csv(MIMIC_III_PATH + "D_ITEMS.csv.gz")
items_iii = items_iii[(items_iii["LABEL"].str.upper().str.contains("PRBC")) | (items_iii["LABEL"].str.upper().str.contains("PACKED RBC"))]
items_iii.groupby("LINKSTO").ITEMID.unique()

In [ ]:
items_iv = pd.read_csv(MIMIC_IV_PATH + "d_items.csv.gz")
items_iv = items_iv[(items_iv["label"].str.upper().str.contains("PRBC")) | (items_iv["label"].str.upper().str.contains("PACKED RBC"))]
items_iv

In [ ]:
transfusion_ids_iii = items_iii.ITEMID.tolist()
transfusion_ids_iv = items_iv.itemid.tolist()

In [ ]:
inputs_iv = pd.read_csv(MIMIC_IV_PATH + "inputevents.csv.gz")
input_mv = pd.read_csv(MIMIC_III_PATH + "INPUTEVENTS_MV.csv.gz", usecols=["SUBJECT_ID","HADM_ID","ICUSTAY_ID","ITEMID","STARTTIME"],
            parse_dates=["STARTTIME"]).rename(columns={"STARTTIME":"CHARTTIME"})
input_cv = pd.read_csv(MIMIC_III_PATH + "INPUTEVENTS_CV.csv.gz", usecols=["SUBJECT_ID","HADM_ID","ICUSTAY_ID","ITEMID","CHARTTIME"],
            parse_dates=["CHARTTIME"])
input_iii = pd.concat([input_mv,input_cv])

In [ ]:
i = input_iii  
print(i[i.ITEMID.isin(transfusion_ids_iii)].HADM_ID.nunique()) #5268
print(i[i.ITEMID.isin(transfusion_ids_iv)].HADM_ID.nunique()) #5268

In [ ]:
12734 - 1759

In [ ]:
items_iv = pd.read_csv(MIMIC_IV_PATH + "procedures_icd.csv.gz")
items_iv[(items_iv["long_title"].str.upper().str.contains("PRBC")) | (items_iv["long_title"].str.upper().str.contains("PACKED RBC"))]

In [ ]:
transfusion_ids =items_iii.ITEMID.tolist()

In [ ]:
transfusion_ids

In [ ]:
import pandas as pd
inputevents_iv = pd.read_csv(MIMIC_IV_PATH + "inputevents.csv.gz")
chartevents_iv = pd.read_csv(MIMIC_IV_PATH + "chartevents.csv.gz", nrows=1000000)


In [ ]:
events = inputevents_iv[inputevents_iv.itemid.isin(transfusion_ids)]
events.head()

In [ ]:
events = chartevents_iv[chartevents_iv.itemid.isin(transfusion_ids)]
events.head()

In [ ]:
pd.read_csv("/Users/zy51nise/Documents/BIONETs/FLabNet/Code/FLabBench-pipeline/data/MIMIC_IV/cohorts/LIT/cohort_pressure_ulcer.csv")

## Check mimic all


In [ ]:
import pandas as pd
mimic_all =pd.read_csv("/Users/zy51nise/Documents/BIONETs/FLabNet/Code_main/flabnet-pipeline/MIMIC_IV/saved_data/cohorts/mimic_all.csv.gz",compression='gzip')
mimic_all_new = pd.read_csv("/Users/zy51nise/Documents/BIONETs/FLabNet/Code_main/FLabBench-pipeline/saved_data/cohorts/LIT/cohort_mimic_all.csv.gz",compression="gzip")

In [ ]:
old_subs= mimic_all.hadm_id.unique()
new_subs = mimic_all_new.hadm_id.unique()

In [ ]:
print(len(old_subs))
print(len(new_subs))
print(set(old_subs)-set(new_subs))

## Test mimic all features

In [ ]:
mimic_all = pd.read_csv("/Users/zy51nise/Documents/BIONETs/FLabNet/Code_main/FLabBench-pipeline/saved_data/features/mimic_all/features.csv.gz")#

In [ ]:
mimic_all.hadm_id.nunique()

In [ ]:
mimic_sub = mimic_all[~mimic_all["subject_id"].isin(cohort_subjects)]

In [ ]:
mimic_sub.hadm_id.nunique()

In [ ]:
mimic_all_old = pd.read_csv("/Users/zy51nise/Documents/BIONETs/FLabNet/Code_main/flabnet-pipeline/MIMIC_IV/saved_data/processed_admission_features_for_ts/mimic_all/mimic_all_admissions_labs_14_days_to_ts.csv.gz")

In [ ]:
mimic_all_old.hadm_id.nunique()

## Test mimic_all folds


In [ ]:
ids = pd.read_csv("/Users/zy51nise/Documents/BIONETs/FLabNet/Code_main/flabnet-pipeline/MIMIC_IV/saved_data/folds/mimic_all/pt_ids.csv")

In [ ]:
len(ids)

In [ ]:
mimic_all_cohort = pd.read_csv("/Users/zy51nise/Documents/BIONETs/FLabNet/Code_main/FLabBench-pipeline/saved_data/cohorts/LIT/cohort_mimic_all.csv.gz")
mimic_all_cohort_old = pd.read_csv("/Users/zy51nise/Documents/BIONETs/FLabNet/Code_main/flabnet-pipeline/MIMIC_IV/saved_data/cohorts/mimic_all.csv.gz")

In [ ]:
mimic_all_cohort


In [ ]:
mimic_all_cohort_old

In [ ]:
import pickle
with open("/Users/zy51nise/Documents/BIONETs/FLabNet/Code_main/flabnet-pipeline/MIMIC_IV/saved_data/folds/mimic_all/fold_0.pkl", "rb") as f:
    fold_0_old = pickle.load(f)
    
    
train_ids_old, val_ids, test_ids = fold_0_old


In [ ]:
print(len(train_ids))
print(len(test_ids))
print(len(val_ids))

In [ ]:

    
    
    
    
train_ids, val_ids, test_ids = fold_0

In [ ]:
print(len(train_ids))
print(len(test_ids))
print(len(val_ids))

In [ ]:
train_ids_old[:,0]

In [ ]:
set(train_ids[:,0]) - set (train_ids_old[:,0])

In [ ]:
409494 + 103764

In [ ]:
cohort_nf = pd.read_csv("/Users/zy51nise/Documents/BIONETs/FLabNet/Code_main/FLabBench-pipeline/saved_data/cohorts/LIT/cohort_neutropenic_fever.csv.gz")
cohort_aplasia = pd.read_csv("/Users/zy51nise/Documents/BIONETs/FLabNet/Code_main/FLabBench-pipeline/saved_data/cohorts/LIT/cohort_aplasia.csv.gz")

In [ ]:
cohort_subjects = pd.concat(
    [cohort_nf["subject_id"], cohort_aplasia["subject_id"]]
).drop_duplicates().to_numpy()

In [ ]:
cohort_subjects

# Check inputdict

In [ ]:
import pickle
with open("/Users/zy51nise/Documents/BIONETs/FLabNet/Code_main/FLabBench-pipeline/saved_data/results/neutropenic_fever/time_series/standard/random_forest/140426/fold_0/grid_none/input_dict.pkl", "rb") as f:
    inputdict = pickle.load(f)



In [ ]:
inputdict.keys()

In [ ]:
inputdict['X_flat']